<a href="https://colab.research.google.com/github/nevil2006/natural-language-processing/blob/main/day_5_pratical_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("suvidyasonawane/spam-vs-ham-emails")

print("Path to dataset files:", path)

100%|██████████| 1.13k/1.13k [00:00<00:00, 1.54MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/suvidyasonawane/spam-vs-ham-emails/versions/1


In [24]:
df = pd.read_csv('/root/.cache/kagglehub/datasets/suvidyasonawane/spam-vs-ham-emails/versions/1/email_spam_dataset.csv')

In [29]:
# Inspect and clean the 'label' column
print("Original unique labels:", df['label'].unique())
df['label'] = df['label'].str.strip()
print("Unique labels after strip():", df['label'].unique())

y = (df['label'] == 'spam').astype(int).values

Original unique labels: ['ham' 'spam']
Unique labels after strip(): ['ham' 'spam']


In [28]:
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess(text, use_lemma=False):
    # Convert non-string types (like float NaN) to string to prevent TypeError
    if not isinstance(text, str):
        text = str(text)

    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower().split()


    words = [w for w in text if w not in stop_words]

    if use_lemma:
        words = [lemmatizer.lemmatize(w) for w in words]
    else:
        words = [ps.stem(w) for w in words]

    return ' '.join(words)

corpus = [preprocess(t, use_lemma=False) for t in df['email_text']]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
cv = CountVectorizer(max_features=2500)
X = cv.fit_transform(corpus).toarray()

In [14]:
tfidf = TfidfVectorizer(max_features=2500)
X = tfidf.fit_transform(corpus).toarray()

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [16]:
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [17]:
y_pred = model.predict(X_test)

In [18]:
accuracy_score(y_test, y_pred)

1.0

In [21]:
tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(corpus).toarray()

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

In [26]:
def evaluate(model, X_te, y_te):
    y_pred = model.predict(X_te)
    print("Accuracy:", accuracy_score(y_te, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_te, y_pred))
    print("Report:\n", classification_report(y_te, y_pred))

# Evaluate Multinomial Naive Bayes model
model = MultinomialNB() # Re-initialize and train NB model
model.fit(X_train, y_train)
evaluate(model, X_test, y_test)

# Initialize and train RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

# Evaluate RandomForestClassifier
evaluate(rf, X_test, y_test)

Accuracy: 1.0
Confusion Matrix:
 [[65]]
Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        65

    accuracy                           1.00        65
   macro avg       1.00      1.00      1.00        65
weighted avg       1.00      1.00      1.00        65



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Accuracy: 1.0
Confusion Matrix:
 [[65]]
Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        65

    accuracy                           1.00        65
   macro avg       1.00      1.00      1.00        65
weighted avg       1.00      1.00      1.00        65



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [ ]:
print("Distribution of classes in y_train:")
print(pd.Series(y_train).value_counts())

print("\nDistribution of classes in y_test:")
print(pd.Series(y_test).value_counts())